# Revised IFC Check Notebook
Notebook setup for comparing and validating transformed IFC/JSON outputs.

In [1]:
from pathlib import Path
import json
import pandas as pd

root = Path('..').resolve()
json_edit_dir = root / 'JSON_Edit'
revised_ifc_dir = root / 'Revised_IFC'
temp_ifc_dir = root / 'Temp_IFC'

print('Root:', root)
print('JSON_Edit exists:', json_edit_dir.exists())
print('Revised_IFC exists:', revised_ifc_dir.exists())
print('Temp_IFC exists:', temp_ifc_dir.exists())

Root: C:\Git\APS-IFC
JSON_Edit exists: True
Revised_IFC exists: True
Temp_IFC exists: True


## Review JSON Object Name, Property, and Value
Load the transformed JSON from `JSON_Edit`, flatten each object's properties, and preview the key columns for quick review.

In [5]:
import re

target_json = json_edit_dir / 'ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json'
assert target_json.exists(), f"JSON file not found: {target_json}"

elements = json.loads(target_json.read_text(encoding='utf-8'))
assert isinstance(elements, list), "Expected top-level JSON array."

rows = []
for obj in elements:
    obj_name = obj.get('Name')
    dbid = obj.get('DbId')
    props = obj.get('Properties', [])
    if isinstance(props, list):
        for prop in props:
            if not isinstance(prop, dict):
                continue
            rows.append({
                'ObjectName': obj_name,
                'DbId': dbid,
                'Property': prop.get('displayName'),
                'Value': prop.get('value')
            })

flat_df = pd.DataFrame(rows)
print(f"Loaded JSON: {target_json}")
print(f"Object count: {len(elements)}")
print(f"Property rows: {len(flat_df)}")

flat_df[['ObjectName', 'DbId', 'Property', 'Value']].head(200)

Loaded JSON: C:\Git\APS-IFC\JSON_Edit\ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json
Object count: 180
Property rows: 3594


,ObjectName,DbId,Property,Value
0,A-1JNL9322340 - Pyramid,5,Name,A-1JNL9322340 - Pyramid
1,A-1JNL9322340 - Pyramid,5,Type,IFCBUILDINGELEMENTPROXY
2,A-1JNL9322340 - Pyramid,5,GUID,8d3567ff-612b-3878-a628-4b9d838df628
3,A-1JNL9322340 - Pyramid,5,Icon,Group
4,A-1JNL9322340 - Pyramid,5,Hidden,No
...,...,...,...,...
195,A-Cable Ladder 90 450,15,ABB_PROJECT_NAME,SSE 2GW Frame Agreement
196,A-Cable Ladder 90 450,15,GLOBALID,iUbr
197,A-Cable Ladder 90 450,15,ABB_UNIT,PC
198,A-Cable Ladder 90 450,15,GLOBALID,153Kd8NCyuvwR3Cg42F3G1


## Check remaining prefixed values
Detect values that still start with an ID-like prefix such as `1JNL9442631_...`.

In [4]:
prefix_pattern = re.compile(r'^\d[A-Za-z0-9]{7,}_.+')

name_prefixed = flat_df[
    flat_df['ObjectName'].astype(str).str.match(prefix_pattern, na=False)
].copy()

value_prefixed = flat_df[
    flat_df['Value'].astype(str).str.match(prefix_pattern, na=False)
].copy()

print(f"Prefixed ObjectName rows: {len(name_prefixed)}")
print(f"Prefixed Value rows: {len(value_prefixed)}")

if len(value_prefixed) > 0:
    print("\nExamples of remaining prefixed property values:")
    display(value_prefixed[['ObjectName', 'DbId', 'Property', 'Value']].head(50))
else:
    print("\nNo remaining prefixed property values detected.")

if len(name_prefixed) > 0:
    print("\nExamples of remaining prefixed object names:")
    display(name_prefixed[['ObjectName', 'DbId', 'Property', 'Value']].head(50))
else:
    print("\nNo remaining prefixed object names detected.")

Prefixed ObjectName rows: 0
Prefixed Value rows: 0

No remaining prefixed property values detected.

No remaining prefixed object names detected.
